# Elementary packing edits: inference-only check
Create a NEW PRIVATE notebook. Upload this notebook and attach a PRIVATE Dataset containing ONLY elementary_pack.json. No previous notebook output or adapter is required. Enable Internet, GPU T4 x2 and the HF_TOKEN Kaggle Secret. Only GPU 0 is exposed. Never paste the token into a cell or Dataset.
12 first-attempt greedy generations on frozen base Gemma 3 4B: two conditions on six previously annotated training paragraphs from six papers. Five new oracle candidates and one reserved diagnostic probe are reported separately. The split case and reserved probe are mandatory; other papers use the shortest compiled prompt. No selection used model outputs or development/test data.
A: concrete name-free style instruction plus source evidence. B: exactly the same instruction and evidence plus reference-derived sentence-packing edits. Claim IDs and grouping are oracle annotation information; this is neither a learned mapper nor deployment/generalization performance. References and target wording are evaluation-only, never generator inputs.
NF4/FP32 on one T4, pinned Gemma/NLI revisions, 768 output tokens and a 4096 total-token budget. The tokenizer audits every prompt before Gemma weights load; overflow fails without truncation. All model parameters are frozen. No training, adapter, retries, generation repairs or reference fallback. Empty and cutoff outputs remain failures.
After generation Gemma unloads before the smaller NLI model loads. Save full semantic diagnostics, source-claim localization matrices, copying, surface style proxies and paired organization results. NLI is fallible, sentence-count matches are not full adherence, and surface style distance is not an author classifier. Minor drift may justify meaningful style gain; binary verifier status does not establish severity or acceptability.
Download the complete elementary_packing_realization_v1 output folder. Keep generations, references and claim evidence PRIVATE. Share only the notebook slug/version for analysis. Phase messages and 30-second heartbeats show progress without revealing credentials.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate==1.14.0" "bitsandbytes==0.50.2" "huggingface-hub==1.33.0" "sentencepiece>=0.2"


In [ ]:
import base64, json, sys, zlib
from pathlib import Path
runtime = Path("/kaggle/working/_elementary_runtime")
files = json.loads(zlib.decompress(base64.b64decode('...')).decode("utf-8"))
for name, source in files.items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(runtime / "src"))


In [ ]:
import importlib
experiment = importlib.import_module("fama_style.rhetoric.elementary_realization")
matches = list(Path("/kaggle/input").rglob("elementary_pack.json"))
if len(matches) != 1:
    raise ValueError("Attach exactly one private elementary_pack.json")
arguments = ["--pack", str(matches[0]), "--expected-pack-sha", '6272cebf16b279be5e3b04341c81a754b65de9c16afd71cb87e13db87e2ba862']
experiment.main(arguments + ["--output-dir", "/kaggle/working/elementary_preflight"])


In [ ]:
experiment.main(arguments + ["--execute-inference", "--output-dir", "/kaggle/working/elementary_packing_realization_v1"])
